## **1. Téléchargement des données de test**

In [ ]:
!wget https://nextcloud.ig.umons.ac.be/s/rYnCgdSmmRerJRa/download/TestData.zip
!unzip TestData.zip -d test_data

## **2. Importation des librairies**

In [ ]:
import torch
from PIL import Image
from pathlib import Path
import pandas as pd
from torchvision import transforms
from tqdm import tqdm
import os
from torch.utils.data import DataLoader
import torchvision.datasets as datasets

# **3. Téléchargement de la BD de test**

In [ ]:
img_size = 224
folder_path  = 'test_data/TestData'

model = torch.jit.load("model_jit.pth")

In [ ]:
test_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor()
    ])

## **4. Générer le fichier CSV de prédictions pour soumission au challenge Kaggle**

In [ ]:
def predict_folder(model, folder_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()


    predictions = []
    filenames = []

    for img_path in os.listdir(folder_path):
        with torch.no_grad():
            # Load and preprocess image
            image = Image.open(folder_path+'/'+img_path).convert('RGB')
            image = test_transform(image).unsqueeze(0).to(device)

            # Get prediction
            output = model(image)
            _, predicted = output.max(1)

            predictions.append(predicted.item())
            filenames.append(img_path)

    # Create submission dataframe
    submission_df = pd.DataFrame({
        'ID': filenames,
        'Label': predictions
    })

    # Save to CSV
    submission_df.to_csv('submission.csv', index=False)
    return submission_df


submission = predict_folder(model, folder_path)